# 06 因子诊断工作台
从 Snapshot 读取因子，执行 IC、分层收益、归因、阈值判断并生成报告。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from alphapurify_bridge.config import load_diagnosis_config, load_factor_registry
from alphapurify_bridge.diagnostics import DiagnosisRunner
from alphapurify_bridge.io import persist_results, write_approved_factors
from alphapurify_bridge.reporting import DiagnosisReporter

## 1. 加载配置并选择因子

In [ ]:
config = load_diagnosis_config()
registry = load_factor_registry()
factor_names = ['dividend_yield']  # 可改为 list(registry['factors'])
start_date = '2015-01-01'
end_date = None
config['diagnosis']['horizons']

## 2. 运行诊断
设 `official=True` 可增加官方 AlphaPurify 1.0.6 抽检；全历史运行会明显更慢。

In [ ]:
runner = DiagnosisRunner(config, registry=registry)
results = runner.diagnose_factors(factor_names, start_date, end_date, official=False)
summary = [{k: r[k] for k in ['factor_name', 'ic_mean', 'ic_ir', 'spread_return', 'status']} for r in results]
summary

## 3. 查看 IC、分层与归因明细

In [ ]:
result = results[0]
result['ic_by_horizon'], result['quantile_returns'], result['checks']

## 4. 生成 Markdown/HTML 报告与 VectorBT 批准清单

In [ ]:
output_root = config['output']['output_dir']
artifacts = persist_results(results, output_root)
reporter = DiagnosisReporter(Path(output_root) / 'reports')
report_paths = [reporter.generate_factor_report(r, fmt) for r in results for fmt in ('md', 'html')]
approved_path = write_approved_factors(results, config['output']['approved_factors_file'], diagnosis_artifact=artifacts['json'])
artifacts, report_paths, approved_path